# Healthcare Utilization & Cost Analytics
## 02. Encounter Analysis

This notebook analyzes how patients use healthcare services and how encounter costs are covered by payers. It builds on the data profiling in `01_data_profiling.ipynb`.

**Questions**
- How many encounters does a typical patient have, and does it differ by gender?
- How does the mix of encounter types change with patient age?
- How does utilization change over time?
- How concentrated is utilization among a small group of patients?
- Who pays for encounters, and how much of the cost do payers cover?

## 1. Import Libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

## 2. Load Data

Only the columns needed for this analysis are loaded. Personal identifiers (names, SSN, address) are not read at all.

In [ ]:
data_dir = Path("../data/raw")

patients = pd.read_csv(
    data_dir / "patients.csv",
    usecols=["Id", "BIRTHDATE", "DEATHDATE", "GENDER", "RACE"],
)
encounters = pd.read_csv(
    data_dir / "encounters.csv",
    usecols=["Id", "START", "STOP", "PATIENT", "PAYER", "ENCOUNTERCLASS",
             "BASE_ENCOUNTER_COST", "TOTAL_CLAIM_COST", "PAYER_COVERAGE"],
)
payers = pd.read_csv(data_dir / "payers.csv", usecols=["Id", "NAME"])

print(f"patients:   {len(patients):,} rows")
print(f"encounters: {len(encounters):,} rows")
print(f"payers:     {len(payers):,} rows")

## 3. Prepare Data

### 3.1 Convert Dates

In [ ]:
patients["BIRTHDATE"] = pd.to_datetime(patients["BIRTHDATE"])
patients["DEATHDATE"] = pd.to_datetime(patients["DEATHDATE"])

# Encounter timestamps carry a time zone; remove it so dates can be compared
encounters["START"] = pd.to_datetime(encounters["START"], utc=True).dt.tz_localize(None)
encounters["STOP"] = pd.to_datetime(encounters["STOP"], utc=True).dt.tz_localize(None)
encounters["YEAR"] = encounters["START"].dt.year

print("Data ends:", encounters["START"].max().date())

### 3.2 Join Patients to Encounters

Each encounter gets the attributes of its patient. `validate="many_to_one"` makes pandas raise an error if a patient `Id` appears more than once, which would silently duplicate encounters.

In [ ]:
enc = encounters.merge(
    patients.rename(columns={"Id": "PATIENT"}),
    on="PATIENT",
    how="left",
    validate="many_to_one",
)

print("Rows before join:", len(encounters))
print("Rows after join: ", len(enc))
print("Encounters without a patient match:", enc["GENDER"].isna().sum())

### 3.3 Age at Encounter

Age is calculated on the date of each encounter, not today, so a patient is counted in the age group they were in at the time of the visit.

In [ ]:
enc["AGE_AT_ENCOUNTER"] = (enc["START"] - enc["BIRTHDATE"]).dt.days / 365.25

print("Negative ages:", (enc["AGE_AT_ENCOUNTER"] < 0).sum())

age_bins = [0, 18, 35, 50, 65, 200]
age_labels = ["0-17", "18-34", "35-49", "50-64", "65+"]
enc["AGE_GROUP"] = pd.cut(enc["AGE_AT_ENCOUNTER"], bins=age_bins, labels=age_labels, right=False)

enc["AGE_GROUP"].value_counts().reindex(age_labels)

### 3.4 Analysis Window

As found in notebook 01, encounter volume is sparse before 2011 and 2020 is incomplete (data ends in April). Time-based analysis therefore uses **2011-2019** only.

In [ ]:
window = enc[enc["YEAR"].between(2011, 2019)].copy()
print(f"Encounters in 2011-2019: {len(window):,} of {len(enc):,}")

## 4. Encounters per Patient

### 4.1 By Gender

In [ ]:
per_patient = enc.groupby("PATIENT").size().rename("encounters").reset_index()
per_patient = per_patient.merge(
    patients.rename(columns={"Id": "PATIENT"})[["PATIENT", "GENDER"]], on="PATIENT"
)

per_patient.groupby("GENDER")["encounters"].agg(["count", "mean", "median", "max"]).round(1)

The distribution is strongly right-skewed (a few patients have very many encounters), so the **median** describes a typical patient better than the mean.

### 4.2 Distribution

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(per_patient["encounters"], bins=40, edgecolor="black")
plt.title("Encounters per Patient")
plt.xlabel("Number of encounters")
plt.ylabel("Number of patients")
plt.tight_layout()
plt.show()

per_patient["encounters"].describe().round(1)

### 4.3 Concentration of Utilization

In [ ]:
counts = per_patient["encounters"].sort_values(ascending=False)
top_n = int(len(counts) * 0.10)

print(f"Top 10% of patients ({top_n} patients) account for "
      f"{counts.head(top_n).sum() / counts.sum() * 100:.1f}% of all encounters")
print(f"Bottom 50% of patients account for "
      f"{counts.tail(len(counts) // 2).sum() / counts.sum() * 100:.1f}% of all encounters")

## 5. Encounter Types by Age Group

In [ ]:
class_by_age = pd.crosstab(enc["AGE_GROUP"], enc["ENCOUNTERCLASS"])
class_by_age_pct = (class_by_age.div(class_by_age.sum(axis=1), axis=0) * 100).round(1)

display(class_by_age)
display(class_by_age_pct)

In [ ]:
class_by_age_pct.plot(kind="bar", stacked=True, figsize=(9, 5))
plt.title("Encounter Type Mix by Age Group")
plt.xlabel("Age at encounter")
plt.ylabel("Share of encounters (%)")
plt.xticks(rotation=0)
plt.legend(title="Encounter class", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## 6. Encounters over Time (2011-2019)

In [ ]:
yearly = window.groupby(["YEAR", "ENCOUNTERCLASS"]).size().unstack(fill_value=0)
yearly["total"] = yearly.sum(axis=1)
yearly

In [ ]:
yearly.drop(columns="total").plot(figsize=(10, 5), marker="o")
plt.title("Encounters per Year by Class (2011-2019)")
plt.xlabel("Year")
plt.ylabel("Number of encounters")
plt.ylim(bottom=0)
plt.legend(title="Encounter class", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## 7. Costs and Payers

### 7.1 Cost per Encounter by Class

Checks whether encounter cost differs between visit types.

In [ ]:
window.groupby("ENCOUNTERCLASS")["TOTAL_CLAIM_COST"].agg(["count", "mean", "min", "max"]).round(2)

### 7.2 Payer Summary

In [ ]:
enc_pay = enc.merge(
    payers.rename(columns={"Id": "PAYER", "NAME": "PAYER_NAME"}),
    on="PAYER",
    how="left",
    validate="many_to_one",
)
print("Encounters without a payer match:", enc_pay["PAYER_NAME"].isna().sum())

by_payer = enc_pay.groupby("PAYER_NAME").agg(
    encounters=("Id", "size"),
    total_cost=("TOTAL_CLAIM_COST", "sum"),
    payer_coverage=("PAYER_COVERAGE", "sum"),
)
by_payer["encounters_pct"] = (by_payer["encounters"] / by_payer["encounters"].sum() * 100).round(1)
by_payer["uncovered_cost"] = by_payer["total_cost"] - by_payer["payer_coverage"]
by_payer["coverage_pct"] = (by_payer["payer_coverage"] / by_payer["total_cost"] * 100).round(1)

by_payer.sort_values("encounters", ascending=False).round(
    {"total_cost": 0, "payer_coverage": 0, "uncovered_cost": 0, "encounters_pct": 1, "coverage_pct": 1}
)

In [ ]:
plot_data = by_payer.sort_values("coverage_pct")

plot_data["coverage_pct"].plot(kind="barh", figsize=(8, 5), title="Share of Claim Cost Covered by Payer")
plt.xlabel("Coverage (%)")
plt.ylabel("")
plt.tight_layout()
plt.show()

### 7.3 Coverage by Encounter Class

Coverage percentage for the five payers with the most encounters.

In [ ]:
top_payers = by_payer.sort_values("encounters", ascending=False).head(5).index

subset = enc_pay[enc_pay["PAYER_NAME"].isin(top_payers)]
coverage_matrix = subset.groupby(["ENCOUNTERCLASS", "PAYER_NAME"])[["PAYER_COVERAGE", "TOTAL_CLAIM_COST"]].sum()
coverage_matrix["coverage_pct"] = coverage_matrix["PAYER_COVERAGE"] / coverage_matrix["TOTAL_CLAIM_COST"] * 100

coverage_matrix["coverage_pct"].unstack("PAYER_NAME").round(1)

## 8. Data Quality Check: Encounters After Death

An encounter should not start after the patient's date of death. A tolerance of one day is allowed because death dates have no time of day.

In [ ]:
after_death = enc[
    enc["DEATHDATE"].notna() & (enc["START"] > enc["DEATHDATE"] + pd.Timedelta(days=1))
].copy()
after_death["days_after_death"] = (after_death["START"] - after_death["DEATHDATE"]).dt.days

print("Encounters after death:", len(after_death))
print("Patients affected:", after_death["PATIENT"].nunique())

after_death[["START", "DEATHDATE", "days_after_death", "ENCOUNTERCLASS"]].sort_values("days_after_death", ascending=False)

### 8.1 Time Zone Hypothesis

Encounter timestamps are stored in UTC, while death dates are plain local dates. An evening visit on the day of death would therefore show a UTC time shortly after midnight on the next day. To test this, shift the timestamps back by 5 hours (US Eastern winter time) and check again.

In [ ]:
shifted_start = after_death["START"] - pd.Timedelta(hours=5)
still_after = (shifted_start > after_death["DEATHDATE"] + pd.Timedelta(days=1)).sum()

print("Encounter start hours (UTC):", sorted(int(h) for h in after_death["START"].dt.hour.unique()))
print("Still after death after a 5-hour shift:", still_after)

## 9. Key Findings

- **Utilization per patient:** A typical patient has 27 encounters (median; mean 45.6). The distribution is strongly right-skewed. The median is 35 for women and 21 for men, while the maximum of 2,006 encounters (one male patient) pulls the male mean (47.1) above the female mean (44.2).
- **Concentration:** The top 10% of patients (117 patients) account for 38.9% of all encounters. The bottom half of patients account for 17.6%.
- **Age and encounter type:** The share of wellness visits falls from 45.3% in children (0-17) to 19.3% in patients 65+. Urgent care rises from about 1% in children to 11.2% in patients 65+. Outpatient visits peak at ages 35-49 (27.8%) and inpatient stays at ages 50-64 (5.8%).
- **Trend:** Encounter volume is stable in 2011-2019, between 2,890 and 3,178 per year, with no clear upward or downward trend.
- **Cost per encounter:** `TOTAL_CLAIM_COST` is $129.16 for every encounter class except inpatient (mean $120.47). Cost per visit therefore does not explain differences between visit types; differences in what is paid come from who pays.
- **Payers:** 19.1% of encounters have no insurance (`NO_INSURANCE`) and no coverage, which makes them the largest uncovered amount (about $1.31M, roughly 2.9 times the uncovered cost of Medicare, the next largest). Among insured encounters coverage ranges from 33% (Dual Eligible) through 51% (Blue Cross Blue Shield) and about 64% (Medicare, Aetna) to 77% (Anthem).
- **Coverage by class:** Wellness visits are covered best (72-90% for the largest insurers). None of the largest insurers shows any coverage for urgent care visits.
- **Data quality:** 9 encounters start on the day after the recorded death date, all within the first three hours after midnight (UTC). The pattern is consistent with UTC timestamps compared against local death dates, and the check in 8.1 tests this.

*Per-patient and payer figures use all years of data; time-based figures use 2011-2019.*
*Encounter claim costs make up only about 2% of all recorded costs (procedures and medications are much larger, see notebook 03), so the payer coverage above describes encounter claims only.*